# Week 9 — Deep Lagrangian Network (Colab)

Trains **DeLaN** (physics-structured inverse dynamics) and compares it head-to-head with the Week-8 MLP and the analytic model.

DeLaN's forward pass differentiates a learned M(q) and V(q) with autograd, so each step is heavier than the MLP. A **GPU runtime helps here** — Runtime → Change runtime type → T4 GPU.

Flow: clone → install → data (reuse or regenerate) → **validate the EL math** → train DeLaN → 3-way comparison → download `delan.pt`.

> Set `REPO_URL` below before running.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Raxy777/robot_arm_delan.git"
REPO_DIR = "robot_arm_delan"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

PROJECT = os.path.abspath(REPO_DIR)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print("working dir:", os.getcwd())

Cloning into 'robot_arm_delan'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 29 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 30.77 KiB | 10.26 MiB/s, done.
Resolving deltas: 100% (2/2), done.
working dir: /content/robot_arm_delan


In [ ]:
!pip install -q mujoco>=3.0
import torch, numpy as np, mujoco
os.environ["MUJOCO_GL"] = "egl"
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

torch: 2.11.0+cu128 | CUDA: True


## Validate the Euler-Lagrange math FIRST
This is torch-free: it feeds the analytic M and V through DeLaN's assembly formula (finite-diff derivatives) and checks it reproduces the true inverse dynamics. If this fails, stop — the bug is in the equations, not the network. If it passes, the assembly is correct and any training issue is optimization, not math.

In [ ]:
!python _verify_week9_math.py

--- DeLaN Euler-Lagrange assembly vs analytic inverse dynamics ---
  samples        = 2000
  max |tau diff| = 5.002e-09 N m  (finite-diff limited, expect ~1e-6)
  PASS (threshold 1e-4)


## Data
Regenerate if `data/` isn't present (repo excludes it). Reuses the exact Week-8 datasets so MLP and DeLaN train on identical data.

In [ ]:
if not os.path.exists("data/train.npz"):
    from data_collection import make_all
    make_all(backend="mujoco", seed=0)
else:
    print("data/ already present — reusing.")

train    :  39000 samples | |qd| max 24.76 | |qdd| max 832.45 | |tau| max  60.00
test_id  :   9750 samples | |qd| max 15.85 | |qdd| max 766.05 | |tau| max  60.00
test_ood :   8000 samples | |qd| max 17.59 | |qdd| max 204.11 | |tau| max  14.53

Saved datasets to /content/robot_arm_delan/data


## (Optional) train the MLP too, so the comparison is complete
Skip if you already downloaded `mlp.pt` from Week 8 into `models/`.

In [ ]:
if not os.path.exists("models/mlp.pt"):
    from train_mlp import train as train_mlp
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_mlp(epochs=300, device=device)
else:
    print("models/mlp.pt present — skipping MLP training.")

epoch    0 | train 0.10784 | val 0.11889
epoch   25 | train 0.00249 | val 0.00287
epoch   50 | train 0.00142 | val 0.00166
epoch   75 | train 0.00068 | val 0.00076
epoch  100 | train 0.00041 | val 0.00050
epoch  125 | train 0.00166 | val 0.00181
epoch  150 | train 0.00038 | val 0.00039
epoch  175 | train 0.00049 | val 0.00063
epoch  200 | train 0.00015 | val 0.00017
epoch  225 | train 0.00016 | val 0.00020
epoch  250 | train 0.00023 | val 0.00029
epoch  275 | train 0.00034 | val 0.00038
epoch  299 | train 0.00120 | val 0.00126

--- Torque prediction RMSE (N m) ---
  in-distribution     : joint1  2.3339 | joint2  1.1767 | overall  1.8482
  OUT-of-distribution : joint1  3.8382 | joint2  1.8948 | overall  3.0267

Expect the OOD error to be much larger than in-distribution — that
gap is the black-box model's Achilles heel, and the motivation for
the physics-structured model in Week 9.

Saved model to /content/robot_arm_delan/models/mlp.pt


## Train DeLaN
Slower than the MLP (autograd assembly per step) and usually wants more epochs. Watch that train/val loss both fall — if val plateaus high, try `lr=3e-4` or more epochs.

In [ ]:
from train_delan import train as train_delan
device = "cuda" if torch.cuda.is_available() else "cpu"
print("training DeLaN on:", device)
delan = train_delan(epochs=600, lr=5e-4, batch=128, device=device)

training DeLaN on: cuda
epoch    0 | train 32.29729 | val 33.91660
epoch   25 | train 1.39103 | val 1.63308
epoch   50 | train 0.42513 | val 0.48018
epoch   75 | train 0.20801 | val 0.21618
epoch  100 | train 0.13696 | val 0.14284
epoch  125 | train 0.06150 | val 0.06480
epoch  150 | train 0.13097 | val 0.14991
epoch  175 | train 0.01453 | val 0.01603
epoch  200 | train 0.00982 | val 0.01177
epoch  225 | train 0.00955 | val 0.01145
epoch  250 | train 0.00520 | val 0.00581
epoch  275 | train 0.00729 | val 0.00760
epoch  300 | train 0.00376 | val 0.00509
epoch  325 | train 0.01110 | val 0.01358
epoch  350 | train 0.00401 | val 0.00483
epoch  375 | train 0.00276 | val 0.00303
epoch  400 | train 0.00179 | val 0.00199
epoch  425 | train 0.01213 | val 0.01330
epoch  450 | train 0.00267 | val 0.00296
epoch  475 | train 0.00363 | val 0.00413
epoch  500 | train 0.00925 | val 0.00956
epoch  525 | train 0.00231 | val 0.00242
epoch  550 | train 0.00144 | val 0.00173
epoch  575 | train 0.00125 | va

## Three-way comparison — the headline table
analytic (upper bound) vs MLP vs DeLaN, in-distribution vs OOD.

In [ ]:
!python evaluate_all.py --backend analytic

Models loaded: analytic, mlp, delan

A. Torque-prediction RMSE (N m)  [open loop]

model              test_id        test_ood
------------------------------------------
analytic            0.0000          0.0000
mlp                 1.8482          3.0267
delan               0.1155          0.6197

B. End-effector RMS tracking error (mm)  [closed loop, analytic]

model          nominal        fast
----------------------------------
analytic         0.007       0.075
mlp              2.730       4.295
delan            0.306       0.519

How to read it: analytic is the upper bound (tiny error everywhere).
A good MLP matches it in-distribution but degrades out-of-distribution.
DeLaN should sit between them out-of-distribution — closer to analytic
than the MLP — because its physics structure constrains extrapolation.


## Download the trained DeLaN model

In [ ]:
from google.colab import files
files.download("models/delan.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>